# `google.colab.ai` pre-flight

*Linguistic Data Analysis II — instructor pre-course check (run once, on the Tohoku Google account students will use).*

**Purpose.** Before building more on top of Colab's built-in Gemini, confirm the backend the day-notebooks call — `from google.colab import ai; ai.generate_text(p)` — is reachable on the Tohoku account, find where (if anywhere) it throttles, and measure how stable the parsed CEFR labels are run-to-run.

Run **Runtime → Run all**. Each step prints a compact report; the final cell summarizes go / no-go.

> This notebook only runs **inside Google Colab** — `google.colab.ai` does not exist elsewhere. If Step 2 fails, the whole `colab.ai` path is unavailable for Tohoku accounts → fall back to the Gemini-API key (D4) and document it.

---

In [ ]:
#@title 📦 Setup — run me first { display-mode: "form" }
# Reuses the SAME backend detection + label parser as the day-notebooks,
# so the numbers below reflect what the course actually runs.
import json, re, time, urllib.request, statistics
from collections import Counter

# --- LLM backend: Colab built-in Gemini (no key). No local fallback here on
#     purpose: this notebook is specifically about the colab.ai path.
from google.colab import ai
generate_text = lambda p: ai.generate_text(p)

def _extract_level(text):
    """Identical to the day-notebooks' parser."""
    m = re.search(r"\b([ABC][12])\b", str(text).upper())
    return m.group(1) if m else "??"

LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
PROMPT = (
    "You are a CEFR rater. Reply with ONLY one label from "
    "A1, A2, B1, B2, C1, C2 for this sentence.\n\nSentence: {text}\nLabel:"
)

# Real CEFR sentences from the course gold set (cycled to reach the call counts).
GOLD_URL = "https://raw.githubusercontent.com/egumasa/linguistic-data-analysis-II-2026/main/sources/resources/datasets/gold/cefr_sentences.json"
gold = json.loads(urllib.request.urlopen(GOLD_URL).read().decode("utf-8"))
print(f"Loaded {len(gold)} gold sentences. Backend: Colab Gemini (colab.ai).")

## Step 2 — Smoke test (is the API even reachable?)

One list-models call + one generate call. If this errors, **stop** — the `colab.ai` path is unavailable on this account.

In [ ]:
#@title Step 2 — smoke test
try:
    print("Available models:")
    try:
        print(" ", ai.list_models())
    except Exception as e:
        print("  (list_models not available:", type(e).__name__, e, ")")
    print("\nSingle call →", repr(generate_text("Say OK")))
    print("\n✅ Reachable.")
except Exception as e:
    print("❌ colab.ai unavailable on this account:", type(e).__name__, e)
    print("   → Escalate to the Gemini-API .env fallback (D4) and stop.")

## Step 3 — Quota ceiling (100-call loop)

Fire 100 real CEFR prompts, one at a time. Each call is wrapped so a rate-limit / quota error is caught and its **index** recorded rather than crashing the run. Reports successes, latency (median / p95), total wall-time, and where (if anywhere) it first throttled.

In [ ]:
#@title Step 3 — 100-call quota + latency probe
N = 100
items = [gold[i % len(gold)] for i in range(N)]

latencies, preds, first_fail, fails = [], [], None, 0
t0 = time.time()
for i, item in enumerate(items, 1):
    prompt = PROMPT.format(text=item["text"])
    c0 = time.time()
    try:
        reply = generate_text(prompt)
        latencies.append(time.time() - c0)
        preds.append(_extract_level(reply))
    except Exception as e:
        fails += 1
        if first_fail is None:
            first_fail = (i, type(e).__name__, str(e)[:200])
        preds.append("ERR")
    if i % 20 == 0:
        print(f"  ...{i}/{N}  ({fails} failures so far)")
wall = time.time() - t0

ok = len(latencies)
print(f"\n{ok}/{N} succeeded, {fails} failed.")
if latencies:
    lat = sorted(latencies)
    p95 = lat[min(len(lat)-1, int(0.95*len(lat)))]
    print(f"latency: median {statistics.median(lat):.2f}s  p95 {p95:.2f}s  "
          f"min {lat[0]:.2f}s  max {lat[-1]:.2f}s")
print(f"total wall-time for {N} calls: {wall:.1f}s "
      f"({wall/N:.2f}s/call incl. overhead)")
print("first failure:", first_fail if first_fail else "none — no throttling in 100 calls")
print("parse rate (non-??/ERR):",
      f"{sum(p in LEVELS for p in preds)}/{N}")

## Step 4 — Extrapolate to ~15 students

The single-user numbers above → what a class of 15 implies. **Caveat:** this measures one account sequentially; it cannot test 15-way *concurrency*. If `colab.ai` quota is per-account (likely), 15 students = 15 independent quotas and Step 3 passing is reassuring. If it's a shared project quota, real load is ~15× and only a multi-account trial can confirm it.

In [ ]:
#@title Step 4 — class-of-15 projection
CLASS = 15
CALLS_PER_STUDENT_PER_DAY = 72 + 72 + 60  # tutorial + few-shot + lab, rough
if latencies:
    per_call = wall / N
    student_day = per_call * CALLS_PER_STUDENT_PER_DAY
    print(f"~{per_call:.2f}s/call → one student\u2019s ~{CALLS_PER_STUDENT_PER_DAY} "
          f"calls/day ≈ {student_day/60:.1f} min of model time.")
    print(f"Class total: {CLASS*CALLS_PER_STUDENT_PER_DAY} calls/day across {CLASS} accounts.")
    print(f"If quota is PER-ACCOUNT: each student runs the {student_day/60:.1f}-min "
          "budget independently → fine.")
    print("If quota is SHARED/project: treat 100-call ceiling from Step 3 as the "
          "whole-class ceiling → likely too low, plan the .env fallback.")
else:
    print("No successful calls in Step 3 — cannot project. Fix Step 2/3 first.")

## Step 5 — Determinism (label stability)

`colab.ai` exposes no temperature knob, so raw text will vary. What matters for the course is whether the **parsed CEFR label** is stable. Send the same prompt 10× for a few sentences and report the modal label and its agreement rate — this is how much run-to-run F1 wobble to warn students about.

In [ ]:
#@title Step 5 — same prompt × 10, label stability
REPEATS = 10
SAMPLES = 4
sample_items = [gold[i * (len(gold)//SAMPLES)] for i in range(SAMPLES)]

print(f"{REPEATS} repeats each, {SAMPLES} sentences:\n")
agreements = []
for item in sample_items:
    prompt = PROMPT.format(text=item["text"])
    labels = []
    for _ in range(REPEATS):
        try:
            labels.append(_extract_level(generate_text(prompt)))
        except Exception as e:
            labels.append("ERR")
    c = Counter(labels)
    modal, modal_n = c.most_common(1)[0]
    agree = modal_n / REPEATS
    agreements.append(agree)
    print(f"  gold {item['label']}  →  modal {modal} "
          f"({modal_n}/{REPEATS} = {agree:.0%})   dist={dict(c)}")

print(f"\nMean label agreement across samples: "
      f"{statistics.mean(agreements):.0%}")
print("→ Tell students their F1 can wobble by roughly the complement of this.")

## Go / no-go

- **Step 2 fails** → `colab.ai` unavailable on Tohoku accounts → use Gemini-API `.env` fallback (D4).
- **Step 3 throttles well before 100** → single-user ceiling too low → fallback / rate-limit guidance.
- **Step 5 agreement low (<~80%)** → warn students & consider majority-vote over repeats in the pipeline.

Record the numbers in `planning/course_planning/prep-plan.md` (milestone 07-09).